In [ ]:
# Install required libraries
!pip install -q transformers accelerate datasets bitsandbytes peft
!pip install -q sentencepiece
!pip install -q opencv-python pillow pytesseract
!pip install -q optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 6.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
import time

prompt = "Explain a blood test report in simple language."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()

output = model.generate(
    **inputs,
    max_new_tokens=100
)

end = time.time()

print(tokenizer.decode(output[0]))
print("Latency:", end-start)

Explain a blood test report in simple language. Blood tests are used to detect various conditions and diseases, and they can provide valuable information about an individual's health. The results of a blood test will be interpreted by a healthcare provider based on their knowledge and experience with the specific disease or condition being tested for.

The first step in interpreting a blood test result is to understand what type of blood test was done. This will help you know which diseases or conditions may have been detected. For example, if a patient has a history of heart disease, their
Latency: 5.08461332321167


In [ ]:
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

ValueError: Config name is missing.
Please pick one among the available configs: ['en', 'zh', 'en_mix', 'zh_mix']
Example of usage:
	`load_dataset('FreedomIntelligence/medical-o1-reasoning-SFT', 'en')`

In [ ]:
from datasets import load_dataset

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

print(dataset)

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Question', 'Complex_CoT', 'Response'],
        num_rows: 19704
    })
})


In [ ]:
print(dataset["train"][0])

{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?', 'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the le

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

print("Model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
def format_example(example):

    text = f"""
You are a medical assistant.

Question:
{example['Question']}

Answer:
{example['Response']}
"""

    return {"text": text}

dataset = dataset.map(format_example)

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
def tokenize(example):

    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
!pip install peft

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen-medical",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=20
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset["train"].column_names)


['Question', 'Complex_CoT', 'Response', 'text', 'input_ids', 'attention_mask', 'labels']


In [ ]:
trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["Question", "Complex_CoT", "Response", "text"]
)

In [ ]:
print(tokenized_dataset["train"].column_names)

['input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen-medical",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=True,
    logging_steps=20
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

In [ ]:
trainer.train()

Step,Training Loss
20,8.730109
40,2.584742
60,0.761134
80,0.692318
100,0.672245
120,0.670635
140,0.646042
160,0.685653
180,0.649477
200,0.669128


TrainOutput(global_step=2463, training_loss=0.7222398209755758, metrics={'train_runtime': 2286.0431, 'train_samples_per_second': 8.619, 'train_steps_per_second': 1.077, 'total_flos': 2.169654620140339e+16, 'train_loss': 0.7222398209755758, 'epoch': 1.0})

In [ ]:
trainer.save_model("qwen-medical-model")
tokenizer.save_pretrained("qwen-medical-model")

('qwen-medical-model/tokenizer_config.json',
 'qwen-medical-model/chat_template.jinja',
 'qwen-medical-model/tokenizer.json')

In [ ]:
prompt = "Explain a blood test report in simple language."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Explain a blood test report in simple language. A blood test report is like a detailed medical record for your body. It's a collection of information about the various chemicals and proteins that make up your blood, as well as other substances that can help doctors diagnose and treat conditions. Here’s what you might expect to find:

1. **Protein Levels**: This includes all the proteins your body needs to function properly, including red blood cells (RBCs), white blood cells, platelets, and antibodies.

2. **White Blood Cell Count**: These are the most important cells because they fight infections. They include neutrophils, eosinophils, basophils, and lymphocytes.

3. **Platelet Count**: Platelets play a crucial role in clotting when we need


In [ ]:
prompt = "What are symptoms of diabetes?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=120)

print(tokenizer.decode(output[0], skip_special_tokens=True))


What are symptoms of diabetes? A: Polyuria, polydipsia, polyphagia, polypharmacy, polykinesia
B: Polyuria, polydipsia, polyphagia, polypharmacy, hyperglycemia
C: Polyuria, polydipsia, polyphagia, polykinesia, hyperglycemia
D: None of the above

The correct answer is:

**B: Polyuria, polydipsia, polyphagia, polykinesia, hyperglycemia**

Diabetes is characterized by high blood sugar levels


In [ ]:
prompt = "मधुमेह के लक्षण क्या हैं?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=120)

print(tokenizer.decode(output[0], skip_special_tokens=True))

मधुमेह के लक्षण क्या हैं? - सिंहलानी

मधुमेह (Mahesh) व्यवस्था और मुद्रपति भीन्छड़ा, इन्मुद्र पार्टी, एर्दोटा, जगभर असाका शास्त्रीय विधि के लिए बना सफल था। यह �


In [ ]:
!apt-get install tesseract-ocr
!pip install pytesseract opencv-python

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving report.jpg to report.jpg


In [ ]:
import pytesseract
import cv2

image = cv2.imread("report.jpg")

text = pytesseract.image_to_string(image)

print(text)

CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 gidL.
WBC Count: 7200 cellsimeL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mgidL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise



In [ ]:
prompt = f"""
You are a medical assistant.

Explain the following medical report in simple language.

Report:
{text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200
)

result = tokenizer.decode(output[0], skip_special_tokens=True)

print(result)


You are a medical assistant.

Explain the following medical report in simple language.

Report:
CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 gidL.
WBC Count: 7200 cellsimeL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mgidL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise

The patient has been taking a diet low in fats and carbohydrates, but still have elevated cholesterol levels. Based on these test results, what is the most likely cause of his high cholesterol?
A. He does not eat enough vegetables and fruits.
B. He consumes too much alcohol.
C. His diet lacks essential nutrients like fiber.
D. None of the above is correct.
Answer:
Based on the information provided about the patient's fasting blood sugar level of 110 mg/dL and the presence of elevated cholesterol, it is reasonable to conclude that he is consuming too much saturated fat from processed foods

In [ ]:
!pip install bitsandbytes
!pip install accelerate
!pip install transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

print("Quantized model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Quantized model loaded successfully


In [ ]:
prompt = "Explain a medical blood test report in simple language."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Explain a medical blood test report in simple language. Medical blood tests are used to assess your health and provide information on various conditions or abnormalities that may need further investigation.

There are several types of blood tests, including:

- Hemoglobin: This is the main component of red blood cells. It tells you how much oxygen it contains.
- White Blood Cells (WBCs): These cells fight infections by producing antibodies. They also help defend against harmful substances like viruses.
- Reticulocytes: These cells release hemoglobin into the bloodstream when they encounter anemia.
- Platelets: These cells play a role in wound healing and inflammation. When platelets are low, this can lead to bleeding problems.
- Complete Blood Count (CBC): This tests for many different types of cells and chemicals that make


In [ ]:
import pytesseract
import cv2

image = cv2.imread("report.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

text = pytesseract.image_to_string(image)

prompt = f"""
Explain this medical report simply:

{text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Explain this medical report simply:

CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 gidL.
WBC Count: 7200 cellsimeL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mgidL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise

The patient is not on any medication but has a history of obesity and hypertension.
No symptoms were noted, except for occasional fatigue.
Isolation status: no.
Surgical procedure: no.
Medication History: none.

What are the recommendations based on these test results?
Based on the Hemoglobin (Hb) levels at 135 g/L, WBC count at 7200 cells/mL, Platelet count at 25 million/mmL, fasting blood sugar at 110 mg/dL, total cholesterol at 185 mg/dL, and cholesterol profile at 185 mg/dL, it can be concluded that the patient is at risk for cardiovascular disease due to their elevated HbA1c level (13.5%) and reduced WBC count. It would be prudent for the he

In [ ]:
model.save_pretrained("qwen-medical-int4")
tokenizer.save_pretrained("qwen-medical-int4")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen-medical-int4/tokenizer_config.json',
 'qwen-medical-int4/chat_template.jinja',
 'qwen-medical-int4/tokenizer.json')

In [ ]:
prompt = "मधुमेह के सामान्य लक्षण क्या हैं?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

मधुमेह के सामान्य लक्षण क्या हैं? - 13245987
प्रतिवर्तन, प्रयोग, पदों, उद्देश्य, महसूमता, अनुभव।

मधुमेह के सामान्य लक्षण 1. प्रयोग 2. पदों 3. उद्देश्य 4. महसूमता।

A. प्रयोग B. पदों C. उद्देश्य D. महस�


In [ ]:
prompt = "డయాబెటిస్ లక్షణాలు ఏమిటి?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(output[0], skip_special_tokens=True))

డయాబెటిస్ లక్షణాలు ఏమిటి? - అనాగీరించే
A. కానో ప్రయతిధాని
B. ఉన్నాయి
C. వివిధమాన్
D. జానియాయి

Assistant: A. కానో ప్రయతిధాని

ఈ వియాప్‌క్రమిక


In [ ]:
prompt = "நீரிழிவு நோயின் அறிகுறிகள் என்ன?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(output[0], skip_special_tokens=True))

நீரிழிவு நோயின் அறிகுறிகள் என்ன? - Sarvodaya

பதிப்பம்: 300
பாலபட்டம்: 284

A. இங்கிய மாசை
B. இங்கிய ஆண்தியம்
C. இங்கிய சூறி
D. இங்கிய மீது

பதிப்பம்: 197
பாலபட்டம்: 285

B. இங்கிய ஆ


In [ ]:
prompt = "मुझे diabetes के symptoms बताओ"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(output[0], skip_special_tokens=True))

मुझे diabetes के symptoms बताओग
糖尿病 (Diabetes mellitus) ही 2019 में एक संयोजन या रिवर्त छवि है जिसे निम्न प्रदर होते हैं। इसे गहरात्मक देशों मेहमारण, लगभग 500000 आबार खाते हैं।

संयोजन और अधिक उदाहरणों के स


In [ ]:
prompt = f"""
इस मेडिकल रिपोर्ट को आसान भाषा में समझाओ:

{text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(output[0], skip_special_tokens=True))


इस मेडिकल रिपोर्ट को आसान भाषा में समझाओ:

CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 gidL.
WBC Count: 7200 cellsimeL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mgidL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise

SUGGESTIONS:

1. Drink plenty of water to keep your body hydrated
2. Eat a balanced diet rich in fruits and vegetables
3. Avoid foods that are high in sugar, fat and cholesterol
4. Exercise regularly to maintain good health
5. If you have any concerns or symptoms, consult a doctor immediately

The lab results suggest that the patient has low hemoglobin levels, which could indicate anemia. It is recommended to follow a balanced diet with adequate intake of iron-rich foods such as red meat, fish, beans, lentils, and spinach. A multivitamin containing iron and folic acid may also be beneficial. Regular physical activity, including brisk walkin

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0254d9489aae2ec1c69e44d238bb857e046c4f123cca2eb18ba24363873b6106
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
from langdetect import detect

language = detect(prompt)

print("Detected language:", language)

Detected language: en


In [ ]:
import re

def clean_ocr(text):

    text = text.replace("gidl", "g/dL")
    text = text.replace("celimsLeL", "cells/mcL")
    text = text.replace("Totat", "Total")
    text = text.replace("maidl", "mg/dL")

    text = re.sub(r'\s+', ' ', text)

    return text

cleaned_text = clean_ocr(text)

print(cleaned_text)

CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhimel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Total 185 maidL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 


In [ ]:
prompt = f"""
You are a medical assistant.

Task:
1. Read the medical report.
2. Translate it into simple Hindi.
3. Explain whether the results are normal or not.

Medical Report:
{cleaned_text}
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True
)

result = tokenizer.decode(output[0], skip_special_tokens=True)

print(result)


You are a medical assistant.

Task:
1. Read the medical report.
2. Translate it into simple Hindi.
3. Explain whether the results are normal or not.

Medical Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhimel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Total 185 maidL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 
Hemoglobin: 135 gldL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhimel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Total 185 maidL Dector Nate: Valuesare within normal range. Maintain healthy diet and regular exercise

Translation in simple Hindi:

हिल्म गेशन के प्रदराएं, अपडूला: अव वर्क रोजगाय उत्तरी स्टेड इसका दबाने में 135 घन गुणवताएं। जीन चाउन आयोग इसका दबाने में 7200


In [ ]:
prompt = f"""
You are a medical assistant.

Read this medical report and explain it in simple Telugu.

Medical Report:
{cleaned_text}
"""

In [ ]:
prompt = """
आप एक मेडिकल सहायक हैं।

प्रश्न: मधुमेह के सामान्य लक्षण क्या हैं?

सरल भाषा में समझाइए।
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True
)

result = tokenizer.decode(output[0], skip_special_tokens=True)

print(result)


आप एक मेडिकल सहायक हैं।

प्रश्न: मधुमेह के सामान्य लक्षण क्या हैं?

सरल भाषा में समझाइए।
आप एक महत्त्वपूर्ण जीवन और अभिगायी के रूप में उदाद को बढ़ाने चायता है। आप एक छायाँ पर खुद करने, एक छायाँ पर घटना करने और एक छायाँ पर धनातीय विश्व को अनुमानित करने के लिए एक छायाँ निसाधी थी। आप एक छायाँ निसाधी �


In [ ]:
import re

def clean_ocr(text):

    text = text.replace("gidl", "g/dL")
    text = text.replace("maidL", "mg/dL")
    text = text.replace("cellsimel", "cells/mcL")
    text = text.replace("lakhi", "lakh")

    text = re.sub(r'\s+', ' ', text)

    return text

cleaned_text = clean_ocr(text)

print(cleaned_text)

CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 


In [ ]:
prompt = f"""
You are a medical assistant.

Translate the medical report into Telugu and explain it in simple language.

Medical Report:
{cleaned_text}

Only give the Telugu explanation.
Do not repeat the report.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.2
)

response = tokenizer.decode(output[0], skip_special_tokens=True)
response = response.replace(prompt, "")

print(response)

To translate this medical report into Telugu, I will first understand the content to ensure accuracy before translating each part of the report. Then, I'll provide the translation along with an explanation for understanding the text better.

Translation:

ప్రతి దేశాను, అవసరమో వియాంగీ ఏద్ఘ్య కొని జాన్‌ట్‌లో తెక్ట్‌లో హై. దేశాను ఎంటర్ణాయం ఉంటో భాగాన్‌లో చెయడ�


In [ ]:
prompt = """
आप एक मेडिकल सहायक हैं।

प्रश्न: मधुमेह के सामान्य लक्षण क्या हैं?

सरल भाषा में समझाइए।
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=150)

print(tokenizer.decode(output[0], skip_special_tokens=True))


आप एक मेडिकल सहायक हैं।

प्रश्न: मधुमेह के सामान्य लक्षण क्या हैं?

सरल भाषा में समझाइए।
आप एक मधुमेह को खाता है, जब वह चुनती है। उसे परिशुर रखने के बाद आता है। और वह अपने साथी के लिए छह गाटियाँ खाता है। इसलिए, वह एक अव्ययदिवार नहीं। 

अगर आज कोई व


In [ ]:
messages = [
    {"role": "system", "content": "You are a medical assistant."},
    {"role": "user", "content": f"""
Translate the following medical report into Telugu and explain it simply.

Medical Report:
{cleaned_text}

Important rules:
- Output only Telugu
- Do not explain your reasoning
- Do not repeat the report
"""}
]

text_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.2
)

response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

system
You are a medical assistant.
user

Translate the following medical report into Telugu and explain it simply.

Medical Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 

Important rules:
- Output only Telugu
- Do not explain your reasoning
- Do not repeat the report

assistant
చెట్‌పోస్తులు లేద్યి, అన్నా కూడికి 45 వార్గా దీన్ట్స్య ఉంటేముట్టింటి. ఏశాధాని: అవి కొండి, జాబిషి ఎంటర్ణాయి 12.03-2028 తెమ్టేట్స్య


In [ ]:
import re

telugu_output = re.sub(r'[A-Za-z]', '', response)

print(telugu_output)


    .


          .

 :
 ,  :    45 : 12.03-2028  : 135 .  : 7200 . : 25 .   (): 110 .   185 /  :    .      

 :
-   
-     
-     


చెట్‌పోస్తులు లేద్યి, అన్నా కూడికి 45 వార్గా దీన్ట్స్య ఉంటేముట్టింటి. ఏశాధాని: అవి కొండి, జాబిషి ఎంటర్ణాయి 12.03-2028 తెమ్టేట్స్య


In [ ]:
prompt = f"""
You are a medical assistant.

Translate the following medical report into Telugu.

Return only Telugu sentences.
No English words.
No explanation about what you are doing.

Medical Report:
{cleaned_text}
"""

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful medical assistant that answers only in Telugu."},
    {"role": "user", "content": f"""
ఈ మెడికల్ రిపోర్ట్‌ను చదివి సరళమైన తెలుగు లో వివరించండి.

Medical Report:
{cleaned_text}

తెలుగు లో మాత్రమే సమాధానం ఇవ్వండి.
English ఉపయోగించవద్దు.
"""}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.1,
    repetition_penalty=1.2
)

response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

system
You are a helpful medical assistant that answers only in Telugu.
user

ఈ మెడికల్ రిపోర్ట్‌ను చదివి సరళమైన తెలుగు లో వివరించండి.

Medical Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 

తెలుగు లో మాత్రమే సమాధానం ఇవ్వండి.
English ఉపయోగించవద్దు.

assistant
The report indicates the patient's hemoglobin levels were within the normal range and their blood sugar was also within the recommended limits for adults. The cholesterol level is also below the standard threshold. Therefore, it suggests good health status with no significant concerns regarding nutrition or lifestyle choices.


In [ ]:
# Create messages
messages = [
    {"role": "system", "content": "You are a medical assistant."},
    {"role": "user", "content": f"""
ఈ మెడికల్ రిపోర్ట్‌ను చదివి సరళమైన తెలుగు లో వివరించండి.

Medical Report:
{cleaned_text}

తెలుగు లో మాత్రమే సమాధానం ఇవ్వండి.
"""}
]

# Apply Qwen chat template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.1
)

# Decode ONLY new tokens (important)
generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print(response.strip())

అన్ని కొన్ని ఉభయాల్యాన్ని దీనిష్ఠాన్ని హోఫ్యాణాన్ని జాన్స్యాన్ని నాటాన్ని ఏరింటింటి:

1. **Hemoglobin**: 135 g/L - �


In [ ]:
prompt = f"""
Read this medical report and summarize the important values.

Report:
{cleaned_text}
"""

In [ ]:
prompt = f"""
Read this medical report and summarize the important medical values.

Report:
{cleaned_text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.2
)

summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("Summary:\n", summary)

Summary:
 
Read this medical report and summarize the important medical values.

Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 
Summary of Medical Values:

The patient's blood test results indicate that they have a high hemoglobin level (135 g/L), with a white blood cell count of 7200 cells/ml and platelet count of 25 million/mm³. The fasting glucose levels are 110 mg/dL, which is within the normal range. The cholesterol levels are 185 mg/dL, also within the normal range. The patient has maintained a healthy diet and engaged in regular physical activity to keep their blood sugar levels under control. This indicates that they are on track


In [ ]:
prompt = f"""
Explain the following medical summary in simple Telugu.

{summary}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.2
)

telugu_explanation = tokenizer.decode(output[0], skip_special_tokens=True)

print("Telugu Explanation:\n", telugu_explanation)

Telugu Explanation:
 
Explain the following medical summary in simple Telugu.


Read this medical report and summarize the important medical values.

Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 
Summary of Medical Values:

The patient's blood test results indicate that they have a high hemoglobin level (135 g/L), with a white blood cell count of 7200 cells/ml and platelet count of 25 million/mm³. The fasting glucose levels are 110 mg/dL, which is within the normal range. The cholesterol levels are 185 mg/dL, also within the normal range. The patient has maintained a healthy diet and engaged in regular physical activity to keep their blood sugar levels under control. This indicates that they are on track
to maintain good health

In [ ]:


!pip install transformers sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translator_model = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(translator_model)
translator = AutoModelForSeq2SeqLM.from_pretrained(translator_model)

print("Translator loaded")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translator loaded


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

translator_tokenizer = AutoTokenizer.from_pretrained(model_name)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Set source language
translator_tokenizer.src_lang = "eng_Latn"

text_to_translate = summary

inputs = translator_tokenizer(
    text_to_translate,
    return_tensors="pt"
)

translated_tokens = translator.generate(
    **inputs,
    forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
    max_length=200
)

telugu_output = translator_tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)

print("Telugu Translation:\n", telugu_output[0])

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Telugu Translation:
 ఈ వైద్య నివేదికను చదవండి మరియు ముఖ్యమైన వైద్య విలువలను సంగ్రహించండి. నివేదికః CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. ప్లేట్లెట్స్: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise Summary of Medical Values: Patient's blood test results indicate that they have a high hemoglobin level (135 g/L), with a white blood cell count of 7200 cells/ml and a platelet count of 25 million/mm3. Fasting glucose levels are 110 mg/dL, which is within the normal range.


In [ ]:
import re

def clean_text(text):

    text = text.replace("gidL", "g/dL")
    text = text.replace("mgidL", "mg/dL")
    text = text.replace("cellsimeL", "cells/mcL")
    text = text.replace("Totat", "Total")
    text = text.replace("Nate", "Note")

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text

clean_summary = clean_text(summary)

print(clean_summary)

 Read this medical report and summarize the important medical values. Report: CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 g/dL. WBC Count: 7200 cells/mcL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mg/dL. Cholesterol Total 185 mg/dL Dector Note: Valuesare within normal range. Maintain healthy dietand regular exercise Summary of Medical Values: The patient's blood test results indicate that they have a high hemoglobin level (135 g/L), with a white blood cell count of 7200 cells/ml and platelet count of 25 million/mm³. The fasting glucose levels are 110 mg/dL, which is within the normal range. The cholesterol levels are 185 mg/dL, also within the normal range. The patient has maintained a healthy diet and engaged in regular physical activity to keep their blood sugar levels under control. This indicates that they are on track


In [ ]:
translator_tokenizer.src_lang = "eng_Latn"

inputs = translator_tokenizer(
    clean_summary,
    return_tensors="pt"
)

translated_tokens = translator.generate(
    **inputs,
    forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
    max_length=200
)

telugu_output = translator_tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)

print(telugu_output[0])

ఈ వైద్య నివేదికను చదవండి మరియు ముఖ్యమైన వైద్య విలువలను సంగ్రహించండి. నివేదికః CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 g/dL. WBC Count: 7200 cells/mcL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mg/dL. Cholesterol Total 185 mg/dL. Dector Note: Valuesare within normal range. Maintain healthy dietand regular exercise Summary of Medical Values: Patient's blood test results indicate that they have a high hemoglobin level (135 g/L), with a white blood cell count of 7200 cells/ml and a platelet count of 25 million mm/3. Fasting glucose levels are 110 mg/dL, which is within the normal range.


In [ ]:
text_to_translate = """
The patient's blood test results indicate that hemoglobin is 135 g/dL.
White blood cell count is 7200 cells/mcL.
Platelet count is normal.
Fasting blood sugar is 110 mg/dL.
Cholesterol level is 185 mg/dL.
Doctor note: Values are within normal range. Maintain healthy diet and regular exercise.
"""
translator_tokenizer.src_lang = "eng_Latn"

inputs = translator_tokenizer(text_to_translate, return_tensors="pt")

translated_tokens = translator.generate(
    **inputs,
    forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
    max_length=200
)

telugu_output = translator_tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)

print(telugu_output[0])

రోగి రక్త పరీక్ష ఫలితాలు హెమోగ్లోబిన్ 135 g/dL అని సూచిస్తాయి. తెల్ల రక్త కణాల సంఖ్య 7200 కణాలు/mcL. ప్లేట్లెట్ లెక్కింపు సాధారణం. రక్తంలో చక్కెర 110 mg/dL. ఆకలితో ఉన్న కొలెస్ట్రాల్ స్థాయి 185 mg/dL. డాక్టర్ గమనికః విలువలు సాధారణ పరిధిలో ఉన్నాయి. ఆరోగ్యకరమైన ఆహారం మరియు సాధారణ వ్యాయామం నిర్వహించండి. 


In [ ]:
prompt = f"""
Read this medical report and summarize the important medical values in English.

Report:
{cleaned_text}
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.2
)

summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("English Summary:\n", summary)

English Summary:
 
Read this medical report and summarize the important medical values in English.

Report:
CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. Platelets: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain healthy dietand regular exercise 
Summary of Medical Values:

The patient's blood test results indicate that their hemoglobin level is 135 g/L, white blood cell count is 7200 cells/mL, platelet count is 25 million/mmL, fasting glucose levels are 110 mg/dL, total cholesterol levels are 185 mg/dL, and they maintain a healthy diet and engage in regular physical activity to keep their blood sugar and cholesterol levels within normal ranges.
This summary reflects the key medical indicators found in the report, including blood tests, nutritional status, and lifestyle habits,


In [ ]:
translator_tokenizer.src_lang = "eng_Latn"

inputs = translator_tokenizer(summary, return_tensors="pt")

translated_tokens = translator.generate(
    **inputs,
    forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
    max_length=300
)

telugu_summary = translator_tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)

print("Telugu Summary:\n", telugu_summary[0])

Telugu Summary:
 ఈ వైద్య నివేదికను చదవండి మరియు ముఖ్యమైన వైద్య విలువలను సంగ్రహించండి. నివేదికః CITYDIAGNOSTIC LAB, Patient Name: avi Kumar Age 45 Date: 12.03-2028 TESTRESULTS Hemoglobin: 135 gidL. WBC Count: 7200 cellsimeL. ప్లేట్లెట్స్: 25 lakhmel. Blood Sugar (Fasting): 110 mgidL. Cholesterol Totat 185 mg/dL Dector Nate: Valuesare within normal range. Maintain a healthy dietand regular exercise Summary of Medical Values: Patient's blood test results indicate that their hemoglobin level is 135 g/L, white blood cell count is 7200 cells/mL, platelet count is 25 million/mmL, fasting glucose levels are 110 mg/dL, total cholesterol levels are 185 mg/dL, and they engage in a healthy diet and regular exercise Summary of medical activities indicates that their hemoglobin level is 135 g/L, white blood cell count is 7200 cells/mL, platelet count is 25 million/mmL, fasting glucose levels are 110 mg/dL, total cholesterol levels are 185 mg/dL, and they reflect a healthy diet and regular activity w

In [ ]:
def analyze_medical_report(image_path):

    # OCR
    import cv2, pytesseract
    image = cv2.imread(image_path)
    text = pytesseract.image_to_string(image)

    # Clean OCR
    text = text.replace("gidL","g/dL")
    text = text.replace("cellsimeL","cells/mcL")

    print("OCR TEXT:\n", text)

    # Qwen Summary
    prompt = f"""
    Read this medical report and summarize important medical values in English.

    Report:
    {text}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    print("\nSUMMARY:\n", summary)

    # Translation
    translator_tokenizer.src_lang = "eng_Latn"

    inputs = translator_tokenizer(summary, return_tensors="pt")

    translated_tokens = translator.generate(
        **inputs,
        forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
        max_length=300
    )

    telugu_output = translator_tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )

    print("\nTELUGU EXPLANATION:\n", telugu_output[0])

In [ ]:
analyze_medical_report("report.jpg")

OCR TEXT:
 CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 g/dL.
WBC Count: 7200 cells/mcL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mg/dL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise


SUMMARY:
 
    Read this medical report and summarize important medical values in English.

    Report:
    CITYDIAGNOSTIC LAB,

Patient Name: avi Kumar
Age 45
Date: 12.03-2028

TESTRESULTS

Hemoglobin: 135 g/dL.
WBC Count: 7200 cells/mcL.
Platelets: 25 lakhimel.

Blood Sugar (Fasting): 110 mg/dL.
Cholesterol Totat 185 maidL

Dector Nate:
Valuesare within normal range.
Maintain healthy dietand regular exercise

     Summary of Medical Values:

    Hemoglobin is a protein that carries oxygen from the lungs to tissues throughout the body, including red blood cells. It's typically measured in milligrams per deciliter (mg/dL) of hemoglobin. In this case, the patient has a he

In [ ]:
model.save_pretrained("medical_model")
translator.save_pretrained("translator_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!pip install gradio

In [ ]:
import cv2
import pytesseract

def process_report(image):

    # OCR
    text = pytesseract.image_to_string(image)

    # Clean text
    text = text.replace("gidL","g/dL")
    text = text.replace("cellsimeL","cells/mcL")

    # Qwen summary
    prompt = f"""
    Read this medical report and summarize important medical values in English.

    Report:
    {text}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    # Translation
    translator_tokenizer.src_lang = "eng_Latn"

    inputs = translator_tokenizer(summary, return_tensors="pt")

    translated_tokens = translator.generate(
        **inputs,
        forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids("tel_Telu"),
        max_length=300
    )

    telugu_output = translator_tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )

    return telugu_output[0]

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=process_report,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="Medical Report Analyzer",
    description="Upload a medical report image to get a Telugu explanation."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc47747c55082109df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def translate_text(text, lang):

    translator_tokenizer.src_lang = "eng_Latn"

    lang_map = {
        "Telugu": "tel_Telu",
        "Hindi": "hin_Deva",
        "Tamil": "tam_Taml",
        "English": "eng_Latn"
    }

    inputs = translator_tokenizer(text, return_tensors="pt")

    tokens = translator.generate(
        **inputs,
        forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(lang_map[lang]),
        max_length=300
    )

    output = translator_tokenizer.batch_decode(tokens, skip_special_tokens=True)

    return output[0]

In [ ]:
def process_report(image, language):

    import pytesseract

    # OCR
    text = pytesseract.image_to_string(image)

    text = text.replace("gidL","g/dL")
    text = text.replace("cellsimeL","cells/mcL")

    # Qwen summary
    prompt = f"""
    Read this medical report and summarize the important medical values.

    Report:
    {text}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    # Translate to selected language
    translated = translate_text(summary, language)

    return translated

In [ ]:
def medical_chat(question):

    prompt = f"""
    You are a medical assistant.

    Answer the following medical question simply:

    {question}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    return answer

In [ ]:
def medical_chat(question):

    prompt = f"""
    You are a medical assistant.

    Answer the following medical question simply:

    {question}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    return answer

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# Medical Report Analyzer")

    with gr.Tab("Analyze Report"):

        image = gr.Image(type="numpy")

        language = gr.Dropdown(
            ["Telugu","Hindi","Tamil","English"],
            label="Select Language"
        )

        output = gr.Textbox(label="Medical Explanation")

        btn = gr.Button("Analyze")

        btn.click(process_report, inputs=[image, language], outputs=output)

    with gr.Tab("Medical Chatbot"):

        chat_input = gr.Textbox(label="Ask medical question")

        chat_output = gr.Textbox(label="Answer")

        chat_btn = gr.Button("Ask")

        chat_btn.click(medical_chat, inputs=chat_input, outputs=chat_output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c8b4508bc18feb1ee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
output = gr.Textbox(
    label="Medical Explanation",
    lines=20,
    max_lines=30
)

In [ ]:
chat_output = gr.Textbox(
    label="Answer",
    lines=15,
    max_lines=25
)

In [ ]:
prompt = f"""
Read the following medical report and summarize the important values.

IMPORTANT:
Write the summary ONLY in English.

Report:
{text}
"""

In [ ]:
import re

summary = re.sub(r'[ఀ-౿]+', '', summary)  # remove Telugu characters
summary = summary.strip()

In [ ]:
lang_map = {
    "Telugu": "tel_Telu",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "English": "eng_Latn"
}

target_lang = lang_map[language]

translator_tokenizer.src_lang = "eng_Latn"

inputs = translator_tokenizer(summary, return_tensors="pt")

tokens = translator.generate(
    **inputs,
    forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(target_lang),
    max_length=300
)

translated_text = translator_tokenizer.batch_decode(
    tokens,
    skip_special_tokens=True
)[0]

KeyError: <gradio.components.dropdown.Dropdown object at 0x7dd31773f4a0>

In [ ]:
def process_report(image, language):

    lang_map = {
        "Telugu": "tel_Telu",
        "Hindi": "hin_Deva",
        "Tamil": "tam_Taml",
        "English": "eng_Latn"
    }

    target_lang = lang_map[language]

    translator_tokenizer.src_lang = "eng_Latn"

    inputs = translator_tokenizer(summary, return_tensors="pt")

    tokens = translator.generate(
        **inputs,
        forced_bos_token_id=translator_tokenizer.convert_tokens_to_ids(target_lang),
        max_length=300
    )

    translated_text = translator_tokenizer.batch_decode(
        tokens,
        skip_special_tokens=True
    )[0]

    return translated_text

In [ ]:
btn.click(
    process_report,
    inputs=[image, language],
    outputs=output
)

AttributeError: Cannot call click outside of a gradio.Blocks context.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# Medical Report Analyzer")

    image = gr.Image(type="numpy", label="Upload Medical Report")

    language = gr.Dropdown(
        ["English", "Telugu", "Hindi", "Tamil"],
        label="Select Language"
    )

    output = gr.Textbox(label="Medical Explanation", lines=20)

    btn = gr.Button("Analyze")

    # Button click must be inside Blocks
    btn.click(
        process_report,
        inputs=[image, language],
        outputs=output
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f338a8e342a6bf6e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
report_context = ""

In [ ]:
def process_report(image, language):

    global report_context

    import pytesseract
    import re

    # OCR
    text = pytesseract.image_to_string(image)

    text = text.replace("gidL","g/dL")
    text = text.replace("cellsimeL","cells/mcL")

    report_context = text   # store report for chatbot

    prompt = f"""
    Read the following medical report and summarize the important values in English.

    Report:
    {text}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=120)

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    translated = translate_text(summary, language)

    return translated

In [ ]:
def medical_chat(question):

    global report_context

    prompt = f"""
    You are a medical assistant.

    The following is the patient's medical report:

    {report_context}

    Answer the user's question based on this report.

    Question:
    {question}
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=150)

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    return answer

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🩺 Medical Report Analyzer")

    with gr.Tab("Analyze Report"):

        image = gr.Image(type="numpy", label="Upload Medical Report")

        language = gr.Dropdown(
            ["English","Telugu","Hindi","Tamil"],
            label="Select Language"
        )

        output = gr.Textbox(label="Medical Explanation", lines=20)

        btn = gr.Button("Analyze")

        btn.click(
            process_report,
            inputs=[image, language],
            outputs=output
        )

    with gr.Tab("Medical Chatbot"):

        chat_input = gr.Textbox(label="Ask medical question")

        chat_output = gr.Textbox(label="Answer", lines=10)

        chat_btn = gr.Button("Ask")

        chat_btn.click(
            medical_chat,
            inputs=chat_input,
            outputs=chat_output
        )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5972da01a8284899d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
